In [2]:
import os
import os.path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from datetime import datetime
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
import base64
sns.set()

In [3]:
datadir = "data"
data = os.path.join(datadir, "nationwide-drugs-fy21-fy24.csv")
df_cbp = pd.read_csv(data)
df_cbp['Area of Responsibility'] = df_cbp['Area of Responsibility'].str.replace(r' FIELD OFFICE| SECTOR', '', regex=True)

In [4]:
df_cbp

,FY,Month (abbv),Component,Region,Land Filter,Area of Responsibility,Drug Type,Count of Event,Sum Qty (lbs)
0,2024,APR,Office of Field Operations,Coastal/Interior,Other,ATLANTA,Cocaine,1,19.224309
1,2024,APR,Office of Field Operations,Coastal/Interior,Other,ATLANTA,Khat (Catha Edulis),2,4.416741
2,2024,APR,Office of Field Operations,Coastal/Interior,Other,ATLANTA,Marijuana,12,161.359636
3,2024,APR,Office of Field Operations,Coastal/Interior,Other,ATLANTA,Other Drugs**,4,6.020383
4,2024,APR,Office of Field Operations,Coastal/Interior,Other,BALTIMORE,Cocaine,2,9.237369
...,...,...,...,...,...,...,...,...,...
9861,2023,SEP,U.S. Border Patrol,Southwest Border,Land Only,TUCSON,Other Drugs**,4,0.222700
9862,2023,SEP,U.S. Border Patrol,Southwest Border,Land Only,YUMA,Fentanyl,2,4.952400
9863,2023,SEP,U.S. Border Patrol,Southwest Border,Land Only,YUMA,Marijuana,11,0.412100
9864,2023,SEP,U.S. Border Patrol,Southwest Border,Land Only,YUMA,Methamphetamine,2,84.452600


In [5]:
drug_group = df_cbp.groupby(["Component", "Region", "Area of Responsibility", "Drug Type"]).sum()

In [6]:
df_cbp["Drug Type"].unique()

array(['Cocaine', 'Khat (Catha Edulis)', 'Marijuana', 'Other Drugs**',
       'Ecstasy', 'Fentanyl', 'Lsd', 'Methamphetamine', 'Heroin',
       'Ketamine'], dtype=object)

In [7]:
drug_group.reset_index(inplace=True)

In [73]:
fig_treemap = px.treemap(drug_group, 
                         path = ["Drug Type", "Area of Responsibility"], 
                         values = 'Sum Qty (lbs)', 
                         custom_data = ["Drug Type", "Region", "Area of Responsibility", "Count of Event", "Sum Qty (lbs)"],
                         color = "Drug Type",
                         color_discrete_map = {
                             "Marijuana": "#012A4A" ,           #1
                             'Methamphetamine': "#01497C",      #2                       
                             'Khat (Catha Edulis)': "#2A6F97",  #3
                             'Cocaine': "#468FAF",              #4
                             'Other Drugs**': "#89C2D9",        #5
                             'Fentanyl': "#013A63",             #6
                             'Ketamine': "#014F86",             #7
                             'Heroin': "#2C7DA0",               #8
                             'Ecstasy': "#61A5C2",              #9        
                             'Lsd': "#A9D6E5"}                  #10
                             )

with open("logo/CBP-logo-blue-lettering.png", "rb") as img:
    encoded_image = base64.b64encode(img.read()).decode()

fig_treemap.add_layout_image(
    dict(
        source=f"data:image/png;base64,{encoded_image}",
        xref="paper", yref="paper",
        x=0.99, y=1.025,
        sizex=0.2, sizey=0.2,
        xanchor="right", yanchor="bottom"
    )
)

fig_treemap.update_traces(
    hovertemplate =
                "<b>Drug Type: </b><br>" +
                "<b>%{customdata[0]}</b><br><br>" +
                "Port of Entry: %{customdata[2]}<br>" +
                "Region: %{customdata[1]}<br>" +
                "Amount seized in pounds (lbs): %{customdata[4]:,.0f}" +
                "<extra></extra>",
    texttemplate = "%{customdata[2]}<br>" +
                   "<b>%{customdata[4]:,.0f} lbs</b><br>",
    textfont=dict(
        size=14,  
        family="Helvetica"
    )
    
)

fig_treemap.update_layout(
    margin=dict(t=150, l=25, r=25, b=50), 
    font_family = "Helvetica",
    width = 1600,
    height = 900,
    title=dict(
        text = "<b>Amount of Drugs seized at U.S. Ports of Entry 2021 - 2024</b>",
        font = dict(
            size=28,
            family = "Helvetica"),
        x = 0.018),
    annotations=[
        dict(
            text="<i>Source: U.S. Customs and Border Protection, FY21 - FY24 Nationwide Drug Seizures</i>",  
            #x=0.034, 
            x=0, 
            y=1.09,  
            font=dict(size=16, family="Helvetica"),  
            showarrow=False 
        ),
        dict(
            text="<b>This treemap visualizes the proportional differences in the quantity of drugs seized by type,</b>" + 
            "<b> with a secondary layer showing the variations in the amount seized at different U.S. ports of entry.</b>",  
            #x=0.034, 
            x=0, 
            y=-0.05,  
            font=dict(size=16, family="Helvetica"),  
            showarrow=False 
        )
    ]
)

fig_treemap.write_html("interactive_plots/bach_nguyen_cbp_drug_distribution.html")